# Tedarikçi Risk Değerlendirmesi Projesi
Bu notebook, veri madenciliği projesi kapsamındaki tüm adımları tek bir çatı altında toplamaktadır:
1. Keşifçi Veri Analizi (EDA)
2. Risk Skorlama ve Sınıflandırma
3. İstatistiksel Testler (VIF, R²)
4. Makine Öğrenmesi Modellemesi (ML)


## Kütüphanelerin Yüklenmesi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")


## Adım 1: Keşifçi Veri Analizi (EDA)

In [ ]:
# 1. Loading the dataset
# Reads the CSV file from the current directory.
file_path = 'supplier_risk.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}\n")
except FileNotFoundError:
    print(f"Error: Could not find {file_path}. Please make sure the file is in the same directory.")
    return

# 2. Showing dataset shape
# Prints the number of rows and columns.
print("--- 2. Dataset Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

# 3. Showing column names and data types
# df.info() provides a concise summary including names, non-null counts, and data types.
print("--- 3. Column Names and Data Types ---")
print(df.info())
print("\n")

# 4. Checking missing values
# Sums the null values for each column and displays only those with missing data.
print("--- 4. Missing Values ---")
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print(missing_values[missing_values > 0])
else:
    print("No missing values found.")
print("\n")

# 5. Checking duplicate rows
# Counts the total number of exact duplicate rows.
print("--- 5. Duplicate Rows ---")
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}\n")

# 6. Showing descriptive statistics
# Provides mean, std, min, max, and quartiles for numerical columns.
print("--- 6. Descriptive Statistics ---")
print(df.describe())
print("\n")

# 7. Showing the distribution of Risk_Category
# Counts the occurrences of each category in the Risk_Category column.
print("--- 7. Distribution of Risk_Category ---")
if 'Risk_Category' in df.columns:
    print(df['Risk_Category'].value_counts())
else:
    print("'Risk_Category' column not found.")
print("\n")

# 8. Showing the distribution of Year
# Counts the number of records for each year and sorts them chronologically.
print("--- 8. Distribution of Year ---")
if 'Year' in df.columns:
    print(df['Year'].value_counts().sort_index())
else:
    print("'Year' column not found.")
print("\n")

# 9. Showing the number of unique suppliers
# Uses nunique() on the Supplier_ID column to find distinct supplier count.
print("--- 9. Number of Unique Suppliers ---")
if 'Supplier_ID' in df.columns:
    print(f"Number of unique suppliers (Supplier_ID): {df['Supplier_ID'].nunique()}")
else:
    print("'Supplier_ID' column not found.")
print("\n")

# 10. Comparing numerical variables by Risk_Category using group means
# Groups by Risk_Category and calculates the mean for all numerical columns.
print("--- 10. Group Means of Numerical Variables by Risk_Category ---")
if 'Risk_Category' in df.columns:
    # Select only numerical columns to avoid errors when calculating means
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        group_means = df.groupby('Risk_Category')[numeric_cols].mean()
        print(group_means)
    else:
        print("No numerical columns found for grouping.")
else:
    print("'Risk_Category' column not found.")
print("\n")

# 11. Creating correlation matrix for numerical variables
# Computes pairwise correlation of numerical columns using Pearson standard correlation coefficient.
print("--- 11. Correlation Matrix ---")
numeric_df = df.select_dtypes(include=[np.number])
if not numeric_df.empty:
    corr_matrix = numeric_df.corr()
    print(corr_matrix)
else:
    print("No numerical variables to compute correlation.")
    corr_matrix = None
print("\n")

# 12. Visualizing
print("--- 12. Generating Visualizations ---")
sns.set_theme(style="whitegrid") # Set a clean visual theme

# 12a. Risk_Category distribution
if 'Risk_Category' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.countplot(
        data=df, 
        x='Risk_Category', 
        order=df['Risk_Category'].value_counts().index, 
        hue='Risk_Category', 
        palette='viridis', 
        legend=False
    )
    plt.title('Distribution of Risk_Category')
    plt.xlabel('Risk Category')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('risk_category_distribution.png')
    print("Görsel kaydedildi: risk_category_distribution.png")
    plt.close()

# 12b. Year distribution
if 'Year' in df.columns:
    plt.figure(figsize=(8, 5))
    # discrete=True ensures bins align with integer years
    sns.histplot(data=df, x='Year', discrete=True, color='skyblue')
    plt.title('Distribution of Year')
    plt.xlabel('Year')
    plt.ylabel('Frequency')
    # Ensure only integer years are shown on the x-axis
    plt.xticks(sorted(df['Year'].unique()))
    plt.tight_layout()
    plt.savefig('year_distribution.png')
    print("Görsel kaydedildi: year_distribution.png")
    plt.close()

# 12c. Boxplots of key numerical variables by Risk_Category
if 'Risk_Category' in df.columns and not numeric_df.empty:
    # Select a few key numerical variables for boxplots (excluding IDs and Year)
    potential_key_vars = [
        'Financial_Stability_Score', 
        'Delivery_Performance_Score', 
        'Quality_Compliance_Score', 
        'MCDM_Score'
    ]
    key_vars = [var for var in potential_key_vars if var in df.columns]
    
    # Fallback if specific expected variables are not in the dataset
    if not key_vars:
        key_vars = [col for col in numeric_df.columns if col not in ['Supplier_ID', 'Year']][:4]

    # Generate a boxplot for each key variable
    for var in key_vars:
        plt.figure(figsize=(8, 5))
        sns.boxplot(
            data=df, 
            x='Risk_Category', 
            y=var, 
            hue='Risk_Category', 
            palette='Set2', 
            legend=False
        )
        plt.title(f'Boxplot of {var} by Risk_Category')
        plt.xlabel('Risk Category')
        plt.ylabel(var)
        plt.tight_layout()
        plt.savefig(f'boxplot_{var}.png')
        print(f"Görsel kaydedildi: boxplot_{var}.png")
        plt.close()

# 12d. Correlation heatmap
if not numeric_df.empty:
    plt.figure(figsize=(12, 10))
    # Exclude IDs like 'Supplier_ID' from the correlation heatmap as they aren't meaningful features
    corr_cols = [col for col in numeric_df.columns if col != 'Supplier_ID']
    corr_matrix_viz = df[corr_cols].corr()
    
    # Draw the heatmap with annotations
    sns.heatmap(
        corr_matrix_viz, 
        annot=True, 
        cmap='coolwarm', 
        fmt=".2f", 
        linewidths=0.5
    )
    plt.title('Correlation Heatmap of Numerical Variables')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png')
    print("Görsel kaydedildi: correlation_heatmap.png")
    plt.close()

## Adım 2: Risk Skorlama Sistemi (Sabit Eşikler)

In [ ]:
# 1. Load the dataset
df = pd.read_csv('supplier_risk.csv')

# Define variables to be used
vars_to_invert = [
    'Financial_Stability_Score',
    'Delivery_Performance_Score',
    'Quality_Compliance_Score',
    'Regulatory_Adherence_Score',
    'Sustainability_Score'
]
vars_direct = [
    'Past_Risk_Level',
    'Incidents_Count'
]
all_vars = vars_to_invert + vars_direct

# 2. Normalize all selected variables to 0-1 range using MinMaxScaler
scaler = MinMaxScaler()
df_normalized = pd.DataFrame(scaler.fit_transform(df[all_vars]), columns=all_vars)

# 3. Convert to risk components
# For performance scores (higher is better), invert them to make them risk components
for col in vars_to_invert:
    df_normalized[col] = 1.0 - df_normalized[col]
    
# vars_direct are already such that higher means higher risk, so we use them directly

# 4. Create Equal-weighted risk score
# Average of all seven risk components
df['Risk_Score_Equal'] = df_normalized.mean(axis=1)

# 5. Create Priority-weighted risk score
df['Risk_Score_Priority'] = (
    0.27 * df_normalized['Financial_Stability_Score'] +
    0.21 * df_normalized['Delivery_Performance_Score'] +
    0.13 * df_normalized['Quality_Compliance_Score'] +
    0.08 * df_normalized['Regulatory_Adherence_Score'] +
    0.11 * df_normalized['Sustainability_Score'] +
    0.10 * df_normalized['Past_Risk_Level'] +
    0.10 * df_normalized['Incidents_Count']
)

# 6. Classify scores
def classify_risk(score):
    if score <= 0.33:
        return 'Low Risk'
    elif score <= 0.66:
        return 'Medium Risk'
    else:
        return 'High Risk'
        
df['Risk_Class_Equal'] = df['Risk_Score_Equal'].apply(classify_risk)
df['Risk_Class_Priority'] = df['Risk_Score_Priority'].apply(classify_risk)

# Set categorical order for meaningful plots and crosstabs
risk_order = ['Low Risk', 'Medium Risk', 'High Risk']
df['Risk_Class_Equal'] = pd.Categorical(df['Risk_Class_Equal'], categories=risk_order, ordered=True)
df['Risk_Class_Priority'] = pd.Categorical(df['Risk_Class_Priority'], categories=risk_order, ordered=True)

# --- Generate Outputs ---

# 1. Descriptive statistics of both risk scores
print("--- 1. Descriptive Statistics of Risk Scores ---")
print(df[['Risk_Score_Equal', 'Risk_Score_Priority']].describe())
print("\n")

# 2. Class distributions for both methods
print("--- 2. Class Distributions ---")
print("Equal-weighted:")
print(df['Risk_Class_Equal'].value_counts().sort_index())
print("\nPriority-weighted:")
print(df['Risk_Class_Priority'].value_counts().sort_index())
print("\n")

# 3. Bar charts comparing Low / Medium / High distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.countplot(data=df, x='Risk_Class_Equal', palette=['#1a9850', '#fee08b', '#d73027'], order=risk_order, legend=False, hue='Risk_Class_Equal')
plt.title('Equal-Weighted Risk Classes')
plt.xlabel('Risk Class')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.countplot(data=df, x='Risk_Class_Priority', palette=['#1a9850', '#fee08b', '#d73027'], order=risk_order, legend=False, hue='Risk_Class_Priority')
plt.title('Priority-Weighted Risk Classes')
plt.xlabel('Risk Class')
plt.ylabel('Count')

plt.tight_layout()
plt.savefig('risk_class_comparison.png')
print("--- 3. Saved 'risk_class_comparison.png' ---")
plt.close()

# 4. Cross-tabulation between Risk_Class_Equal and Risk_Class_Priority
print("\n--- 4. Cross-tabulation (Equal vs Priority) ---")
crosstab = pd.crosstab(df['Risk_Class_Equal'], df['Risk_Class_Priority'], rownames=['Equal Weight'], colnames=['Priority Weight'])
print(crosstab)
print("\n")

# 6. Save the new dataset
output_file = 'supplier_risk_scored.csv'
df.to_csv(output_file, index=False)
print(f"--- 6. Saved new dataset to '{output_file}' ---")

## Adım 3: Risk Skorlama (Kuantil/Yüzdelik Eşikler)

In [ ]:
# Load the scored dataset
df = pd.read_csv('supplier_risk_scored.csv')

# 1. Calculate quantile thresholds
quantiles_equal = df['Risk_Score_Equal'].quantile([0.33, 0.66])
quantiles_priority = df['Risk_Score_Priority'].quantile([0.33, 0.66])

print("--- 1. Quantile Threshold Values ---")
print(f"Risk_Score_Equal thresholds: 33% = {quantiles_equal[0.33]:.4f}, 66% = {quantiles_equal[0.66]:.4f}")
print(f"Risk_Score_Priority thresholds: 33% = {quantiles_priority[0.33]:.4f}, 66% = {quantiles_priority[0.66]:.4f}")
print("\n")

# Save thresholds to CSV
thresholds_df = pd.DataFrame({
    'Scoring_Method': ['Risk_Score_Equal', 'Risk_Score_Priority'],
    'Low_Medium_Threshold': [quantiles_equal[0.33], quantiles_priority[0.33]],
    'Medium_High_Threshold': [quantiles_equal[0.66], quantiles_priority[0.66]]
})
thresholds_df.to_csv('quantile_thresholds.csv', index=False)
print("--- Saved 'quantile_thresholds.csv' ---\n")

# 2. Assign quantile-based classes
def classify_quantile(score, q33, q66):
    if score <= q33:
        return 'Low Risk'
    elif score <= q66:
        return 'Medium Risk'
    else:
        return 'High Risk'
        
df['Risk_Class_Equal_Quantile'] = df['Risk_Score_Equal'].apply(
    lambda x: classify_quantile(x, quantiles_equal[0.33], quantiles_equal[0.66])
)
df['Risk_Class_Priority_Quantile'] = df['Risk_Score_Priority'].apply(
    lambda x: classify_quantile(x, quantiles_priority[0.33], quantiles_priority[0.66])
)

# Set categorical order
risk_order = ['Low Risk', 'Medium Risk', 'High Risk']
df['Risk_Class_Equal_Quantile'] = pd.Categorical(df['Risk_Class_Equal_Quantile'], categories=risk_order, ordered=True)
df['Risk_Class_Priority_Quantile'] = pd.Categorical(df['Risk_Class_Priority_Quantile'], categories=risk_order, ordered=True)

# Also ensure old ones are ordered
df['Risk_Class_Equal'] = pd.Categorical(df['Risk_Class_Equal'], categories=risk_order, ordered=True)
df['Risk_Class_Priority'] = pd.Categorical(df['Risk_Class_Priority'], categories=risk_order, ordered=True)

# 2. Class distributions
print("--- 2. Class Distributions ---")
print("Equal-weighted (Fixed vs Quantile):")
print("Fixed:")
print(df['Risk_Class_Equal'].value_counts().sort_index())
print("\nQuantile:")
print(df['Risk_Class_Equal_Quantile'].value_counts().sort_index())

print("\nPriority-weighted (Fixed vs Quantile):")
print("Fixed:")
print(df['Risk_Class_Priority'].value_counts().sort_index())
print("\nQuantile:")
print(df['Risk_Class_Priority_Quantile'].value_counts().sort_index())
print("\n")

# 3. Bar charts comparing both thresholding approaches
plt.figure(figsize=(14, 10))
sns.set_theme(style="whitegrid")

palette = ['#1a9850', '#fee08b', '#d73027']

plt.subplot(2, 2, 1)
sns.countplot(data=df, x='Risk_Class_Equal', palette=palette, order=risk_order, legend=False, hue='Risk_Class_Equal')
plt.title('Equal-Weighted (Fixed Threshold)')
plt.xlabel('')
plt.ylabel('Count')

plt.subplot(2, 2, 2)
sns.countplot(data=df, x='Risk_Class_Equal_Quantile', palette=palette, order=risk_order, legend=False, hue='Risk_Class_Equal_Quantile')
plt.title('Equal-Weighted (Quantile Threshold)')
plt.xlabel('')
plt.ylabel('Count')

plt.subplot(2, 2, 3)
sns.countplot(data=df, x='Risk_Class_Priority', palette=palette, order=risk_order, legend=False, hue='Risk_Class_Priority')
plt.title('Priority-Weighted (Fixed Threshold)')
plt.xlabel('Risk Class')
plt.ylabel('Count')

plt.subplot(2, 2, 4)
sns.countplot(data=df, x='Risk_Class_Priority_Quantile', palette=palette, order=risk_order, legend=False, hue='Risk_Class_Priority_Quantile')
plt.title('Priority-Weighted (Quantile Threshold)')
plt.xlabel('Risk Class')
plt.ylabel('Count')

plt.tight_layout()
plt.savefig('threshold_comparison.png')
print("--- 3. Saved 'threshold_comparison.png' ---")
plt.close()

# 4. Cross-tabulation between fixed-threshold and quantile-threshold classes
print("\n--- 4. Cross-tabulation (Fixed vs Quantile) ---")
print("\nEqual-Weighted:")
print(pd.crosstab(df['Risk_Class_Equal'], df['Risk_Class_Equal_Quantile'], rownames=['Fixed'], colnames=['Quantile']))

print("\nPriority-Weighted:")
print(pd.crosstab(df['Risk_Class_Priority'], df['Risk_Class_Priority_Quantile'], rownames=['Fixed'], colnames=['Quantile']))
print("\n")

# 5. Save the updated dataset
output_file = 'supplier_risk_scored_quantile.csv'
df.to_csv(output_file, index=False)
print(f"--- 5. Saved updated dataset to '{output_file}' ---")

## Adım 4: İstatistiksel Geçerlilik Testleri (VIF, R²)

In [ ]:
# Load dataset
df = pd.read_csv('supplier_risk_scored_quantile.csv')

# Define features
features = [
    'Financial_Stability_Score',
    'Delivery_Performance_Score',
    'Quality_Compliance_Score',
    'Regulatory_Adherence_Score',
    'Sustainability_Score',
    'Past_Risk_Level',
    'Incidents_Count'
]
X = df[features]

# --- 1. Calculate VIF (Variance Inflation Factor) ---
print("========== VIF Analysis ==========")
X_vif = add_constant(X)
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data[vif_data["Feature"] != "const"].reset_index(drop=True)

print(vif_data.to_string(index=False))
vif_data.to_csv("vif_results.csv", index=False)

# --- 2. Calculate R2 Score ---
print("\n========== R2 Score Analysis ==========")
# For Equal Weighted Score
y_equal = df['Risk_Score_Equal']
lr_equal = LinearRegression()
lr_equal.fit(X, y_equal)
y_pred_equal = lr_equal.predict(X)
r2_equal = r2_score(y_equal, y_pred_equal)

# For Priority Weighted Score
y_priority = df['Risk_Score_Priority']
lr_priority = LinearRegression()
lr_priority.fit(X, y_priority)
y_pred_priority = lr_priority.predict(X)
r2_priority = r2_score(y_priority, y_pred_priority)

print(f"R2 Score for predicting Risk_Score_Equal: {r2_equal:.4f}")
print(f"R2 Score for predicting Risk_Score_Priority: {r2_priority:.4f}")

with open("r2_results.txt", "w") as f:
    f.write(f"R2 Score for predicting Risk_Score_Equal: {r2_equal:.4f}\n")
    f.write(f"R2 Score for predicting Risk_Score_Priority: {r2_priority:.4f}\n")

## Adım 5: Makine Öğrenmesi Modellemesi

In [ ]:
def train_and_evaluate(X, y, target_name, file_suffix):
    # Encode target
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
    )
    
    # Define models with appropriate pipelines
    models = {
        'Logistic Regression': Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=1000, random_state=42))
        ]),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(random_state=42),
        'K-Nearest Neighbors': Pipeline([
            ('scaler', StandardScaler()),
            ('model', KNeighborsClassifier())
        ])
    }
    
    results = []
    
    # Prepare figure for confusion matrices
    plt.figure(figsize=(15, 10))
    
    for i, (name, model) in enumerate(models.items(), 1):
        # Train and predict
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Precision_Macro': prec,
            'Recall_Macro': rec,
            'F1_Macro': f1
        })
        
        # Plot confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.subplot(2, 2, i)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=le.classes_, yticklabels=le.classes_)
        plt.title(f'{name}\n({target_name})')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        
    plt.tight_layout()
    plt.savefig(f'confusion_matrices_{file_suffix}.png')
    plt.close()
    
    # Save comparison table
    results_df = pd.DataFrame(results)
    results_df.to_csv(f'model_comparison_{file_suffix}.csv', index=False)
    
    # Feature importances for Random Forest
    rf_model = models['Random Forest']
    importances = rf_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances[indices], y=[X.columns[i] for i in indices], 
                palette='magma', hue=[X.columns[i] for i in indices], legend=False)
    plt.title(f'Random Forest Feature Importance\n({target_name})')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig(f'rf_feature_importance_{file_suffix}.png')
    plt.close()

    return results_df

def main():
    # Load dataset
    df = pd.read_csv('supplier_risk_scored_quantile.csv')
    
    # Define feature sets
    features_base = [
        'Financial_Stability_Score',
        'Delivery_Performance_Score',
        'Quality_Compliance_Score',
        'Regulatory_Adherence_Score',
        'Sustainability_Score',
        'Past_Risk_Level',
        'Incidents_Count'
    ]
    features_year = features_base + ['Year']
    
    feature_sets = {
        'without_year': features_base,
        'with_year': features_year
    }
    
    # Define targets
    targets = {
        'priority_fixed': ('Priority Fixed (Primary)', 'Risk_Class_Priority'),
        'equal_fixed': ('Equal Fixed (Primary)', 'Risk_Class_Equal'),
        'priority_quantile': ('Priority Quantile (Secondary)', 'Risk_Class_Priority_Quantile'),
        'equal_quantile': ('Equal Quantile (Secondary)', 'Risk_Class_Equal_Quantile')
    }
    
    all_results = {}
    
    # Iterate through combinations
    for target_key, (target_name, target_col) in targets.items():
        y = df[target_col]

        for fs_name, fs_cols in feature_sets.items():
            X = df[fs_cols]

            print(f"\n========== {target_name} | {fs_name.replace('_', ' ').title()} ==========")
            suffix = f"{target_key}_{fs_name}"
            
            # Train models and save outputs
            res_df = train_and_evaluate(X, y, f"{target_name} ({fs_name})", suffix)
            print(res_df[['Model', 'F1_Macro']])
            
            all_results[f"{target_name}_{fs_name}"] = res_df
            
    # Final Comparison Interpretation
    print("\n=================================================================")
    print("FINAL COMPARISON: YEAR vs NO YEAR (F1 Macro Scores)")
    print("=================================================================\n")
    
    for target_name, _ in targets.values():
        print(f"--- {target_name} ---")
        res_no_year = all_results[f"{target_name}_without_year"]
        res_with_year = all_results[f"{target_name}_with_year"]
        
        for model in ['Logistic Regression', 'Random Forest']:
            f1_no = res_no_year[res_no_year['Model'] == model]['F1_Macro'].values[0]
            f1_yes = res_with_year[res_with_year['Model'] == model]['F1_Macro'].values[0]
            diff = f1_yes - f1_no
            print(f"{model.ljust(20)} | Without Year: {f1_no:.4f} | With Year: {f1_yes:.4f} | Diff: {diff:+.4f}")
        print("")

    print("--- Interpretation ---")
    print("Does 'Year' add meaningful predictive value?")
    print("Comparing the F1 scores shows that adding 'Year' generally provides negligible ")
    print("or zero improvement to the predictive performance of the models.")
    print("Since our risk classes were derived from the performance metrics (which are ")
    print("time-agnostic), the contextual variable 'Year' acts mostly as noise.")
    print("Therefore, excluding 'Year' produces a more robust, generalizable model ")
    print("without sacrificing predictive accuracy.")

main()